#### Install libraries



In [ ]:
from loguru import logger
import sys

logger.remove()
# Add a new loguru handler to only show messages at ERROR level or higher
logger.add(sys.stderr, level="ERROR")

!pip install "spacy<3.8.0,>=3.7.4"
!python -m spacy download en_core_web_md

import spacy
nlp = spacy.load("en_core_web_md")

In [ ]:
import spacy
nlp = spacy.load("en_core_web_md")

#### Authenticate on Google

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

#### Call BigQuery

In [ ]:
from google.cloud import bigquery
project_id = 'ai-on-healthcare'
client = bigquery.Client(project=project_id)

#### Subjects with native coronary artery

In [ ]:
#Atherosclerosis of native coronary artery
ICD_FILTER = ['41401']

diagnose_query = """
    SELECT *
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    LIMIT 10000
"""

# Run the query and convert the results to a Pandas DataFrame
df_diagnoses = client.query(diagnose_query).to_dataframe()
df_diagnoses = df_diagnoses[df_diagnoses['ICD9_CODE'].isin(ICD_FILTER)]
subj_filtered = df_diagnoses['SUBJECT_ID'].unique()
print(subj_filtered)

#### Query Notes for those subject (limited to 10000 notes)

In [ ]:
notes_query = """
    SELECT *
    FROM `physionet-data.mimiciii_notes.noteevents`
    LIMIT 10000
"""
# Run the query and convert the results to a Pandas DataFrame
df_notes = client.query(notes_query).to_dataframe()
df_notes_filtered = df_notes[df_notes['SUBJECT_ID'].isin(subj_filtered)]
df_notes_filtered.to_parquet("/content/notes_filtered.parquet")   # add this
df_notes_filtered.head(3)

#### function for plotting t-sne (taken from professor Ying class)

In [ ]:
import numpy as np

def tsne_plot(model,words, preTrained=False):
    "Creates and TSNE model and plots it"
    labels = []
    tokens = []

    for word in words:
      if preTrained:
          tokens.append(model[word])
      else:
          tokens.append(model.wv[word])
      labels.append(word)

    tokens = np.array(tokens)
    tsne_model = TSNE(perplexity=50, early_exaggeration=12, n_components=2, init='pca', n_iter=1000, random_state=23)
    new_values = tsne_model.fit_transform(tokens)

    x = []
    y = []
    for value in new_values:
        x.append(value[0])
        y.append(value[1])

    plt.figure(figsize=(16, 16))
    for i in range(len(x)):
        plt.scatter(x[i],y[i])
        plt.annotate(labels[i],
                     xy=(x[i], y[i]),
                     xytext=(5, 2),
                     textcoords='offset points',
                     ha='right',
                     va='bottom')
    plt.show()

#### Graph entities embedding by using UMAP

In [ ]:
!pip install umap-learn
import umap

def umap_plot(model, words, preTrained=False):
    "Creates a UMAP model and plots it"
    labels = []
    tokens = []

    for word in words:
        if preTrained:
            tokens.append(model[word])
        else:
            tokens.append(model.wv[word])
        labels.append(word)

    tokens = np.array(tokens)
    umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                           metric='cosine', random_state=23)
    new_values = umap_model.fit_transform(tokens)

    x = []
    y = []
    for value in new_values:
        x.append(value[0])
        y.append(value[1])

    plt.figure(figsize=(16, 16))
    for i in range(len(x)):
        plt.scatter(x[i], y[i])
        plt.annotate(labels[i],
                     xy=(x[i], y[i]),
                     xytext=(5, 2),
                     textcoords='offset points',
                     ha='right',
                     va='bottom')
    plt.show()

#### Using Displacy for showing entities

In [ ]:
from spacy import displacy

# First note's text from the filtered DataFrame
for i in range(0, len(df_notes_filtered)):
  note = df_notes_filtered['TEXT'].iloc[i]

  # Run it through whichever model you want to inspect
  doc = nlp(note)

# Render inline in the notebook with highlighted entity spans
  displacy.render(doc, style="ent", jupyter=True)

#### Creating corpus of entities extracted from SpaCy

In [ ]:
corpus=[]
for row in range(0, len(df_notes_filtered)):
  str_tokens=[]
  tokens= nlp(df_notes_filtered.iloc[row]['TEXT']).ents
  for i in range(0, len(tokens)):
    str_tokens.append(tokens[i].text)
  corpus.append(list(str_tokens))

corpus_flat = [tok for note in corpus for tok in note]
print(len(corpus_flat))

#### Graph corpus embedding with t-SNE for SpaCy

In [ ]:
!pip install gensim
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
model1 = Word2Vec(corpus, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

#### Graph corpus embedding using UMAP

In [ ]:
umap_plot(model1,new_v)

#### Corpus of entities extracted from SciSpaCy

In [ ]:
import sys
print(sys.executable)
!{sys.executable} -m pip install scispacy==0.5.4
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
import spacy
nlp = spacy.load("en_ner_bc5cdr_md")
corpus=[]
for row in range(0, len(df_notes_filtered)):
  str_tokens=[]
  tokens= nlp(df_notes_filtered.iloc[row]['TEXT']).ents
  for i in range(0, len(tokens)):
    str_tokens.append(tokens[i].text)
  corpus.append(list(str_tokens))

corpus_flat = [tok for note in corpus for tok in note]
print(len(corpus_flat))

#### Graph SciSpacy Corpus usign t-SNE

In [ ]:
model1 = Word2Vec(corpus, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

#### Graph SciSpacy Corpus usign UMAP

In [ ]:
umap_plot(model1,new_v)

#### Corpus of entities extracted from medSpaCy

In [ ]:

#import medspacy

nlp = spacy.load("en_ner_bc5cdr_md")
nlp.add_pipe("medspacy_context")

corpus = []
for doc in nlp.pipe(df_notes_filtered['TEXT'], batch_size=50):
    str_tokens = [ent.text for ent in doc.ents
                  if not ent._.is_negated
                  and not ent._.is_family
                  and not ent._.is_hypothetical]
    corpus.append(str_tokens)

corpus_flat = [tok for note in corpus for tok in note]
print(len(corpus_flat))

#### Graph MedSpaCy Corpus by usign t-SNE

In [ ]:
model_med = Word2Vec(corpus, min_count=1)
vocabs_med = model_med.wv.key_to_index.keys()
new_v = np.array(list(vocabs_med))
tsne_plot(model1, new_v)

#### Graph MedSpaCy Corpus by usign UMAP







In [ ]:
umap_plot(model1,new_v)